# LangChain 核心模块学习：Chains

对于简单的大模型应用，单独使用语言模型（LLMs）是可以的。

**但更复杂的大模型应用需要将 `LLMs` 和 `Chat Models` 链接在一起 - 要么彼此链接，要么与其他组件链接。**

LangChain 为这种“链式”应用程序提供了 `Chain` 接口。

LangChain 以通用方式定义了 `Chain`，它是对组件进行调用序列的集合，其中可以包含其他链。

In [1]:
! pip install -U langchain

Defaulting to user installation because normal site-packages is not writeable


## LLMChain

LLMChain 是 LangChain 中最简单的链，作为其他复杂 Chains 和 Agents 的内部调用，被广泛应用。

一个LLMChain由PromptTemplate和语言模型（LLM or Chat Model）组成。它使用直接传入（或 memory 提供）的 key-value 来规范化生成 Prompt Template（提示模板），并将生成的 prompt （格式化后的字符串）传递给大模型，并返回大模型输出。

![](../images/llm_chain.png)

## Router Chain: 实现条件判断的大模型调用


这段代码构建了一个可定制的链路系统，用户可以提供不同的输入提示，并根据这些提示获取适当的响应。

主要逻辑：从`prompt_infos`创建多个`LLMChain`对象，并将它们保存在一个字典中，然后创建一个默认的`ConversationChain`，最后创建一个带有路由功能的`MultiPromptChain`。

![](../images/router_chain.png)

In [ ]:
from langchain.chains.router import MultiPromptChain
from langchain_openai import OpenAI, ChatOpenAI
from langchain.chains import ConversationChain
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch

In [3]:
physics_template = """你是一位非常聪明的物理教授。
你擅长以简洁易懂的方式回答关于物理的问题。
当你不知道某个问题的答案时，你会坦诚承认。

这是一个问题：
{input}"""


math_template = """你是一位很棒的数学家。你擅长回答数学问题。
之所以如此出色，是因为你能够将难题分解成各个组成部分，
先回答这些组成部分，然后再将它们整合起来回答更广泛的问题。

这是一个问题：
{input}"""

In [4]:
prompt_infos = [
    {
        "name": "物理",
        "description": "适用于回答物理问题",
        "prompt_template": physics_template,
    },
    {
        "name": "数学",
        "description": "适用于回答数学问题",
        "prompt_template": math_template,
    },
]

In [ ]:
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.7)

In [ ]:
# 使用 LCEL 创建目标链字典
destination_chains = {}

# 遍历prompt_infos列表，为每个信息创建一个LCEL链。
for p_info in prompt_infos:
    name = p_info["name"]  # 提取名称
    prompt_template = p_info["prompt_template"]  # 提取模板
    # 创建PromptTemplate对象
    prompt = PromptTemplate(template=prompt_template, input_variables=["input"])
    # 使用 LCEL 的管道运算符创建链
    chain = prompt | llm | StrOutputParser()
    # 将新创建的chain对象添加到destination_chains字典中
    destination_chains[name] = chain

# 创建一个默认链用于处理无法分类的查询
default_prompt = PromptTemplate(
    template="回答以下问题：\n{input}",
    input_variables=["input"]
)
default_chain = default_prompt | llm | StrOutputParser()

In [ ]:
type(default_chain)

### 使用 LLMRouterChain 实现条件判断调用

这段代码定义了一个chain对象（LLMRouterChain），该对象首先使用router_chain来决定哪个destination_chain应该被执行，如果没有合适的目标链，则默认使用default_chain。

In [ ]:
# 使用 LCEL 实现路由逻辑
from langchain_core.prompts import ChatPromptTemplate
import json

In [ ]:
# 从prompt_infos中提取目标信息并将其转化为字符串列表
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
# 使用join方法将列表转化为字符串，每个元素之间用换行符分隔
destinations_str = "\n".join(destinations)

# 创建路由提示模板
router_template = """根据用户的问题，选择最合适的专家来回答。

可选的专家：
{destinations}

问题：{{input}}

请返回 JSON 格式：{{"destination": "专家名称", "next_inputs": "问题"}}
如果问题不适合任何专家，返回 {{"destination": "DEFAULT", "next_inputs": "问题"}}
"""

router_prompt = ChatPromptTemplate.from_template(
    router_template.format(destinations=destinations_str)
)

In [10]:
print(destinations)

['物理: 适用于回答物理问题', '数学: 适用于回答数学问题']


In [11]:
print(destinations_str)

物理: 适用于回答物理问题
数学: 适用于回答数学问题


In [12]:
print(MULTI_PROMPT_ROUTER_TEMPLATE)

Given a raw text input to a language model select the model prompt best suited for the input. You will be given the names of the available prompts and a description of what the prompt is best suited for. You may also revise the original input if you think that revising it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt names specified below OR it can be "DEFAULT" if the input is not well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (must include ```json at the start of the respon

In [13]:
print(router_template)

Given a raw text input to a language model select the model prompt best suited for the input. You will be given the names of the available prompts and a description of what the prompt is best suited for. You may also revise the original input if you think that revising it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}
```

REMEMBER: "destination" MUST be one of the candidate prompt names specified below OR it can be "DEFAULT" if the input is not well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
物理: 适用于回答物理问题
数学: 适用于回答数学问题

<< INPUT >>
{input}

<< OUTPUT (must include ```json at the start of the

In [ ]:
# 使用 LCEL 创建路由链
def route_question(info):
    """根据路由结果选择合适的链"""
    if isinstance(info, str):
        try:
            # 尝试从字符串中提取 JSON
            import re
            json_match = re.search(r'\{[^}]+\}', info)
            if json_match:
                info = json.loads(json_match.group())
            else:
                return default_chain.invoke({"input": info})
        except:
            return default_chain.invoke({"input": info})
    
    destination = info.get("destination", "DEFAULT")
    next_input = info.get("next_inputs", info.get("input", ""))
    
    print(f"\n选择的专家: {destination}")
    print(f"处理的问题: {next_input}\n")
    
    if destination in destination_chains:
        return destination_chains[destination].invoke({"input": next_input})
    else:
        return default_chain.invoke({"input": next_input})

# 创建完整的路由链
chain = router_prompt | llm | StrOutputParser() | route_question

In [ ]:
print(chain.invoke({"input": "黑体辐射是什么？"}))

In [ ]:
print(
    chain.invoke({"input": "大于40的第一个质数是多少，使得这个质数加一能被3整除？"})
)

In [ ]:
# Verbose mode is handled differently in LCEL
# router_chain.verbose = True

In [ ]:
print(chain.invoke({"input": "黑洞是什么？"}))

### Homework

#### 扩展 Demo：实现生物、计算机和汉语言文学老师 PromptTemplates 及对应 Chains